In [15]:
import numpy as np
from scipy.stats import norm, multivariate_normal
from scipy.optimize import minimize
from scipy.special import expit

## **Gaussian Copula**

$$
\begin{array}{ccl}
\Phi(x) &:& \text{standard normal CDF} \quad \rightarrow \quad \texttt{norm.cdf(x)}
\\
\Phi^{-1}(u) &:& \text{standard normal inverse CDF}\quad \rightarrow \quad \texttt{norm.ppf(u)}
\\
\phi(x) &:& \text{standard normal density}\quad \rightarrow \quad \texttt{norm.pdf(x)}
\\
\Phi_\rho(x,y) &:& \text{bivariate standard normal CDF with correlation } \rho \quad \rightarrow \quad \texttt{multivariate\_normal.cdf([x, y], mean, cov)}
\\
\phi_\rho(x,y) &:& \text{bivariate standard normal density with correlation } \rho \quad \rightarrow \quad \texttt{multivariate\_normal.pdf([x, y], mean, cov)}
\end{array}
$$

 **Gaussian Copula Density $c_\rho$**

$$
\begin{array}{ccl}
c_\rho(u,v) &=& \displaystyle\frac{\mathcal{N}_2\{\Phi^{-1}(u),\Phi^{-1}(v)|0,1,\rho\}}{\mathcal{N}\{\Phi^{-1}(u)|0,1\}\mathcal{N}\{\Phi^{-1}(v)|0,1\}} \\
&=& \displaystyle\frac{1}{\sqrt{1-\rho^2}} \exp\left[-\displaystyle\frac{\rho^2(z_u^2+z_v^2)-2\rho z_u z_v}{2(1-\rho^2)}\right]
\end{array}
$$

where
$$
z_u = \Phi^{-1}(u),
\qquad
z_v = \Phi^{-1}(v)
$$

In [15]:
def gaussian_copula_density(u, v, rho):

    u = np.clip(u, 1e-6, 1 - 1e-6)
    v = np.clip(v, 1e-6, 1 - 1e-6)
    rho = np.clip(rho, -1 + 1e-6, 1 - 1e-6)

    z_u = norm.ppf(u)
    z_v = norm.ppf(v)

    numerator = np.exp( - (rho**2 * (z_u**2 + z_v**2) - 2 * rho * z_u * z_v) / (2 * (1 - rho**2)))
    denominator = np.sqrt(1 - rho**2)

    return numerator / denominator

 **Gaussian Copula CDF** $C_\rho$

$$
C_\rho(u,v)
= \Phi_\rho\left(\Phi^{-1}(u),\Phi^{-1}(v)\right)
$$

where
$$
z_u = \Phi^{-1}(u), 
\qquad 
z_v = \Phi^{-1}(v)
$$

In [14]:
def gaussian_copula_cdf(u, v, rho):

    u = np.clip(u, 1e-6, 1 - 1e-6)
    v = np.clip(v, 1e-6, 1 - 1e-6)
    rho = np.clip(rho, -1 + 1e-6, 1 - 1e-6)

    z_u = norm.ppf(u)
    z_v = norm.ppf(v)

    mean = [0, 0]
    cov = [[1, rho], [rho, 1]]

    return multivariate_normal.cdf([z_u, z_v], mean=mean, cov=cov)


**Conditional Gaussian Copula CDF** $H_\rho$

$$
H_\rho(u,v)
=
\Phi
\left(
\frac{
\Phi^{-1}(u)-\rho\Phi^{-1}(v)
}{
\sqrt{1-\rho^2}
}
\right)
$$

In [13]:
def gaussian_conditional_copula_cdf(u, v, rho):

    u = np.clip(u, 1e-6, 1 - 1e-6)
    v = np.clip(v, 1e-6, 1 - 1e-6)
    rho = np.clip(rho, -1 + 1e-6, 1 - 1e-6)

    z_u = norm.ppf(u)
    z_v = norm.ppf(v)

    return norm.cdf((z_u - rho * z_v) / np.sqrt(1 - rho**2))

**Multivariate Gaussian Copula Density $c_R$**

For $d$-dimensional $u=(u_1,\ldots,u_d)$,

$$
c_R(u_1,\ldots,u_d)
=
\frac{\phi_R(z_1,\ldots,z_d)}
{\prod_{j=1}^{d}\phi(z_j)}
$$

where

$$
z_j = \Phi^{-1}(u_j), \qquad j=1,\ldots,d.
$$

Equivalently,

$$
c_R(u)
=
|R|^{-1/2}
\exp\left[
-\frac{1}{2}
z^\top (R^{-1}-I)z
\right],
$$

where

$$
z=(z_1,\ldots,z_d)^\top,
\qquad
R \text{ is the correlation matrix.}
$$

In [1]:
def gaussian_copula_density_multivariate(u, R):

    eps = 1e-6

    u = np.asarray(u)
    u = np.clip(u, eps, 1 - eps)

    z = norm.ppf(u)

    d = len(z)

    R = np.asarray(R)

    sign, logdet = np.linalg.slogdet(R)

    if sign <= 0:
        return 1e-12

    R_inv_z = np.linalg.solve(R, z)

    exponent = -0.5 * (z @ R_inv_z - z @ z)

    log_density = -0.5 * logdet + exponent

    density = np.exp(log_density)

    density = max(density, 1e-12)

    return density

In [4]:
u = np.array([0.3, 0.7, 0.5])

R = np.array([
    [1.0, 0.5, 0.5],
    [0.5, 1.0, 0.5],
    [0.5, 0.5, 1.0]
])

gaussian_copula_density_multivariate(u, R)

np.float64(1.074201604922916)

## **Vine copula**

In [18]:
import pyvinecopulib as pv

$$
\text{vine}_1\quad:\quad \hat c_{vine}^{(1)}
\left(U_{t+1}^{(1)}, U_t,\ldots , U_{t-k+1}\right)
$$

$$
\text{vine}_2\quad:\quad\hat c_{vine}^{(2)}
\left(U_{t+1}^{(2)}, U_t, \ldots, U_{t-k+1}\right)
$$

$$
\vdots
$$


$$
\text{vine}_k\quad:\quad \hat c_{vine}^{(k)}
\left(U_{t+1}^{(3)}, U_t, \ldots, U_{t-k+1}\right)
$$

In [3]:
def fit_conditional_vines(U_condition, U_target):
    
    U_condition = np.asarray(U_condition)
    U_target = np.asarray(U_target)

    d = U_target.shape[1]

    vines = []
    for j in range(d):

        data_j = np.column_stack([U_target[:, j], U_condition])
        
        vine_j = pv.Vinecop.from_data(data_j)
        vines.append(vine_j)

    return vines

$$
\widehat{c}^{(j)}
\left(u \mid u_{\text{condition}}\right)
=\frac{\widehat{c}^{(j)}
\left(u, u_{\text{condition}}\right)
}{
\displaystyle\int_0^1\widehat{c}^{(j)}\left(s, u_{\text{condition}}\right)\, ds}
$$

In [1]:
def conditional_vine_copula_pdf(vine, u, u_condition, n_grid=300):

    eps = 1e-6
    u = np.clip(u, eps, 1 - eps)
    u_condition = np.asarray(u_condition)

    s_grid = np.linspace(eps, 1 - eps, n_grid)

    numerator_input = np.concatenate([[u], u_condition]).reshape(1, -1)
    numerator = vine.pdf(numerator_input)[0]

    denom_inputs = np.column_stack([s_grid, np.tile(u_condition, (n_grid, 1))])

    denom_values = vine.pdf(denom_inputs) 
    denominator = np.trapz(denom_values, s_grid) # integral

    cond_pdf = numerator / denominator
    cond_pdf = max(cond_pdf, 1e-12)

    return cond_pdf

$$
\widehat{C}^{(j)}
\left(
u \mid u_{\text{condition}}
\right)
=\frac{\displaystyle\int_0^u\widehat{c}^{(j)}\left(s, u_{\text{condition}}\right)\, ds
}{\displaystyle\int_0^1\widehat{c}^{(j)}\left(s, u_{\text{condition}}\right)\, ds}
$$

In [14]:
def conditional_vine_copula_cdf(vine, u, u_condition, n_grid=300):
    
    eps = 1e-6
    u = np.clip(u, eps, 1 - eps)
    u_condition = np.asarray(u_condition)

    # denominator
    s_full = np.linspace(eps, 1 - eps, n_grid)

    denom_inputs = np.column_stack([s_full, np.tile(u_condition, (n_grid, 1))])

    denom_values = vine.pdf(denom_inputs)
    denominator = np.trapz(denom_values, s_full)

    # numerator
    s_part = np.linspace(eps, u, n_grid)

    numer_inputs = np.column_stack([s_part, np.tile(u_condition, (n_grid, 1))])

    numer_values = vine.pdf(numer_inputs)
    numerator = np.trapz(numer_values, s_part)

    cond_cdf = numerator / denominator
    cond_cdf = np.clip(cond_cdf, 0.0, 1.0)

    return cond_cdf

## **Time-varying Copula**

**Patton dynamic gaussian copula**

**GAS model**